## Análise Seguros e Sinistros

Perguntas:
- Correlação Prêmio x Risco: 
    - Os clientes que pagam os maiores prêmios (acima do 3º quartil) têm, proporcionalmente, menos sinistros do que os que pagam menos (abaixo do 1º quartil)? Ou seja, o preço está ajustado corretamente ao risco?
- Influência do Capital Segurado:
    - Existe uma tendência de que seguros com capitais mais altos (proteção maior) tenham uma frequência de acidentes diferente dos seguros de capital baixo?
- Perfil Demográfico:
     - O gênero do contratante (SEXO) influencia a razão entre número de seguros e número de acidentes? Um grupo é estatisticamente mais "seguro" que o outro?
- Geografia:
    -  Qual REGIAO apresenta a pior sinistralidade (maior razão de acidentes por número de apólices)?


### Vamos carregar o dataset

In [0]:
import os

current_path = os.getcwd()
repo_name = "Grupo7-Setor-de-Seguros"

if repo_name in current_path:
    root_path = current_path.split(repo_name)[0] + repo_name
else:
    root_path = os.path.dirname(os.path.dirname(os.getcwd()))

caminho_arquivo = f"{root_path}/data/processed/prata/seguros_sinistros.csv"

print(f"Diretório Raiz: {root_path}")
print(f"Arquivo Alvo: {caminho_arquivo}")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, BooleanType, DateType, LongType

schema_seguros = StructType([
    StructField("nome_contratante", StringType(), True),
    StructField("estado_contratante", StringType(), True),
    StructField("data_contratacao", DateType(), True),
    StructField("valor_pagamento", DoubleType(), True),
    StructField("valor_premio", DoubleType(), True),
    StructField("nome_beneficiario", StringType(), True),
    StructField("status_apolice", StringType(), True),
    StructField("Cobertura1", StringType(), True),
    StructField("Valor Cob 1", DoubleType(), True),
    StructField("Cobertura2", StringType(), True),
    StructField("Valor Cob 2", DoubleType(), True),
    StructField("Cobertura3", StringType(), True),
    StructField("Valor Cob 3", DoubleType(), True),
    StructField("capital_segurado", DoubleType(), True),
    StructField("tipo_sinistro", StringType(), True),
    StructField("valor_sinistro", DoubleType(), True),
    StructField("quem_forma_beneficiados", StringType(), True),
    StructField("status_seguro", StringType(), True),
    StructField("regiao_sinistro", StringType(), True),
    StructField("REGIAO", StringType(), True),
    StructField("SEXO", StringType(), True),
    StructField("TRIMESTRE", IntegerType(), True),
    StructField("ACIMA_DE_3_QUARTIL_PREMIO", BooleanType(), True),
    StructField("ABAIXO_DE_1_QUARTIL_PREMIO", BooleanType(), True),
    StructField("ACIMA_DE_3_QUARTIL_CAPITAL", BooleanType(), True),
    StructField("ABAIXO_DE_1_QUARTIL_CAPITAL", BooleanType(), True),
    StructField("QTD_ACIDENTES_POR_NOME_SEGURADO", LongType(), True),
    StructField("QTD_ACIDENTES_POR_NOME_CONTRATANTE", LongType(), True),
    StructField("RAZÃO_PAGAMENTO_PREMIO", DoubleType(), True),
    StructField("RAZÃO_PAGAMENTO_CAPITAL", DoubleType(), True)
])


df_seguros = spark.read \
    .format("csv") \
    .schema(schema_seguros) \
    .option("header", "true") \
    .option("sep", ",") \
    .option("dateFormat", "yyyy-MM-dd") \
    .load(caminho_arquivo)

df_seguros.printSchema()
display(df_seguros)

---
Criamos uma função para analisar a razão entre número de acidentes e número de seguros.

In [0]:
from pyspark.sql import functions as F

col_acidente = F.when(F.col("regiao_sinistro") == "null", 1).otherwise(0)

expr_razao = (F.sum(col_acidente) / F.count("*")).alias("razao_seguros_acidentes")

def calcular_razao(df_input, grupo_cols=None):
    if grupo_cols:
        resultado = df_input.groupBy(grupo_cols).agg(
            F.sum(col_acidente).alias("total_acidentes"),
            F.count("*").alias("total_seguros"),
            expr_razao
        )
    else:
        resultado = df_input.agg(
            F.sum(col_acidente).alias("total_acidentes"),
            F.count("*").alias("total_seguros"),
            expr_razao
        )
    return resultado

---
### Comparamos a razão entre sinistros e seguros
- do conjunto como um todo
- de cada sexo
- de cada região

In [0]:
# --- A. Conjunto como um todo ---
display(calcular_razao(df_seguros))

# --- B. Por Sexo ---
display(calcular_razao(df_seguros, "SEXO"))

# --- C. Por Região ---
display(calcular_razao(df_seguros, "REGIAO"))

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

### Resultado

1. Sexo
- Percebe-se que os homens tem maior razão, mas não muito significativo (cerca de 4% mais)

2. Região
- A razão é praticamente igual em todas as regiões, embora haja uma grande diferença nos números totais (região Sul contratando 8k seguros e região Nordeste 23k)
---

#### Comparamos as razões e os valores de prêmio e capital
- dividimos os valores de capital e prêmio mais altos (acima 3 quartil) 
- dividimos os valores de capital e prêmio mais baixos (abaixo 1 quartil) 

In [0]:
# --- D. Capital Segurado ---
print("Razão para Capital ALTO (Acima Q3/Q4):")
df_capital_alto = df_seguros.filter(F.col("ACIMA_DE_3_QUARTIL_CAPITAL") == True)
display(calcular_razao(df_capital_alto))

print("Razão para Capital BAIXO (Abaixo Q1):")
df_capital_baixo = df_seguros.filter(F.col("ABAIXO_DE_1_QUARTIL_CAPITAL") == True)
display(calcular_razao(df_capital_baixo))

In [0]:
# --- E. Valor do Prêmio ---
print("Razão para Prêmio ALTO (Acima Q3/Q4):")
df_capital_alto = df_seguros.filter(F.col("ACIMA_DE_3_QUARTIL_PREMIO") == True)
display(calcular_razao(df_capital_alto))

print("Razão para Prêmio BAIXO (Abaixo Q1):")
df_capital_baixo = df_seguros.filter(F.col("ABAIXO_DE_1_QUARTIL_PREMIO") == True)
display(calcular_razao(df_capital_baixo))

---
### Resultado

- Percebe-se que todos os grupos têm praticamente a mesma razão entre sinistros e seguros. Além disso, há grande semelhança no número de seguros totais em todos os grupos.
- Com isso, conclui-se que os valores de capital e prêmio, analisados em separado, não predizem a ocorrência de sinistros.
---

### Comparamos razão sinistros-acidentes com razão pagamento-prêmio e pagamento-capital

- Como o pagamento e o prêmio separadamente não influenciam a quantidade de sinistros por seguros, vamos analisar a proporcionalidade entre pagamento-prêmio e pagamento-capital para ver se eles influenciam no número de acidentes.


In [0]:
def filtrar_e_calcular_por_quartil(df, coluna_numerica):
    # O parametro 0.01 é a precisão do erro relativo (quanto menor, mais preciso e mais lento)
    quartis = df.stat.approxQuantile(coluna_numerica, [0.25, 0.75], 0.01)
    q1 = quartis[0]
    q3 = quartis[1]
    
    print(f"Analisando coluna: {coluna_numerica} | Q1: {q1}, Q3: {q3}")
    
    df_q1 = df.filter(F.col(coluna_numerica) < q1)
    df_q3 = df.filter(F.col(coluna_numerica) > q3)
    
    display(calcular_razao(df_q1))
    display(calcular_razao(df_q3))

In [0]:
# --- F. Razão Pagamento-Prêmio ---
filtrar_e_calcular_por_quartil(df_seguros, "RAZÃO_PAGAMENTO_PREMIO")

# --- G. Razão Pagamento-Capital ---
filtrar_e_calcular_por_quartil(df_seguros, "RAZÃO_PAGAMENTO_CAPITAL")

---
### Resultado

- Assim como antes, parece que os valores do prêmio e capital não influenciam na quantidade de sinitros dos contratos de seguros.
---

---
## Resultado Geral


1. Correlação Prêmio x Risco:
- O risco é praticamente o mesmo para os maiores e menores prêmios.

2. Influência do Capital Segurado:
- A frequência de acidentes é a mesma para capitais segurados mais altos e mais baixos.

3. Perfil Demográfico:
- Há uma leve preponderância do sexo masculino, o que indica que análises mais complexas (controlando por idade, região ou outros parâmetros) podem identificar grupos de clientes significativemente mais custosos.


4. Geografia:
- As razões são muito similares entre as regiões. Mas há uma grande diferença entre o número total de seguros contratados de algumas regiões.
- Análises que controlem por outros parâmetros podem identificar padrões mais claros de contratação entre diferentes regiões.


### Tabela Resumo

In [0]:
from pyspark.sql import functions as F
from functools import reduce

def calcular_metricas_base(df_grouped):
    col_acidente = F.when(F.col("regiao_sinistro") == "null", 1).otherwise(0)
    return df_grouped.agg(
        F.sum(col_acidente).alias("Qtd_Acidentes"),
        F.count("*").alias("Qtd_Seguros"),
        (F.sum(col_acidente) / F.count("*")).alias("Razao_Seguro_Acidente")
    )

def padronizar_df(df, nome_categoria, col_segmento_origem=None, nome_segmento_fixo=None):
    # Se for um agrupamento (ex: Sexo)
    if col_segmento_origem:
        df_padrao = df.withColumnRenamed(col_segmento_origem, "Segmento")
    # Se for um filtro (ex: Q3)
    else:
        df_padrao = df.withColumn("Segmento", F.lit(nome_segmento_fixo))
    
    # Adiciona a categoria da análise e seleciona ordem final
    return df_padrao.withColumn("Categoria_Analise", F.lit(nome_categoria)) \
                    .select("Categoria_Analise", "Segmento", "Qtd_Seguros", "Qtd_Acidentes", "Razao_Seguro_Acidente")


dfs_para_unir = []

df_geral = calcular_metricas_base(df_seguros.groupBy()) # groupBy vazio = tudo
dfs_para_unir.append(padronizar_df(df_geral, "Geral", nome_segmento_fixo="Total"))

# B. Por Gênero
df_sexo = calcular_metricas_base(df_seguros.groupBy("SEXO"))
dfs_para_unir.append(padronizar_df(df_sexo, "Demografia", col_segmento_origem="SEXO"))

# C. Por Região
df_regiao = calcular_metricas_base(df_seguros.groupBy("REGIAO"))
dfs_para_unir.append(padronizar_df(df_regiao, "Geografia", col_segmento_origem="REGIAO"))

# D. Por Quartis de Capital
df_cap_alto = calcular_metricas_base(df_seguros.filter("ACIMA_DE_3_QUARTIL_CAPITAL = true").groupBy())
dfs_para_unir.append(padronizar_df(df_cap_alto, "Perfil Capital", nome_segmento_fixo="Alto Capital (>Q3)"))
df_cap_baixo = calcular_metricas_base(df_seguros.filter("ABAIXO_DE_1_QUARTIL_CAPITAL = true").groupBy())
dfs_para_unir.append(padronizar_df(df_cap_baixo, "Perfil Capital", nome_segmento_fixo="Baixo Capital (<Q1)"))

# E. Por Quartis de Prêmio
df_prem_alto = calcular_metricas_base(df_seguros.filter("ACIMA_DE_3_QUARTIL_PREMIO = true").groupBy())
dfs_para_unir.append(padronizar_df(df_prem_alto, "Perfil Prêmio", nome_segmento_fixo="Alto Prêmio (>Q3)"))
df_prem_baixo = calcular_metricas_base(df_seguros.filter("ABAIXO_DE_1_QUARTIL_PREMIO = true").groupBy())
dfs_para_unir.append(padronizar_df(df_prem_baixo, "Perfil Prêmio", nome_segmento_fixo="Baixo Prêmio (<Q1)"))

df_final_consolidado = reduce(lambda df1, df2: df1.unionByName(df2), dfs_para_unir)

# Exibição
display(df_final_consolidado)